# MariaDB to PostgreSQL Schema Conversion

This notebook converts the `pafr_database_schema.sql` dump from MariaDB/MySQL syntax into PostgreSQL-compatible DDL suitable for Supabase. It normalizes quoting, data types, engine/charset clauses, enum definitions, and unsupported checks.

In [ ]:
from pathlib import Path
import re

schema_path = Path(r'c:\\wamp64\\www\\PAFR2.0\\PAFR\\pafr_database_schema.sql')
output_path = schema_path.with_name('pafr_database_schema_postgres.sql')

print('Schema path:', schema_path)
print('Output path:', output_path)

In [ ]:
raw_sql = schema_path.read_text(encoding='utf-8')
print('Loaded SQL length:', len(raw_sql))

In [ ]:
sql = raw_sql.replace('`', '')
sql = sql.replace('_utf8mb4\'', '\'')
sql = re.sub(r'bigint\(20\)\s+NOT\s+NULL\s+AUTO_INCREMENT', 'bigint GENERATED ALWAYS AS IDENTITY', sql, flags=re.IGNORECASE)
sql = re.sub(r'int\(11\)\s+NOT\s+NULL\s+AUTO_INCREMENT', 'integer GENERATED ALWAYS AS IDENTITY', sql, flags=re.IGNORECASE)
sql = re.sub(r'\bAUTO_INCREMENT\b', '', sql, flags=re.IGNORECASE)
sql = re.sub(r'\btinyint\(1\)\b', 'boolean', sql, flags=re.IGNORECASE)
sql = re.sub(r'\bdatetime\b', 'timestamp', sql, flags=re.IGNORECASE)
sql = re.sub(r'current_timestamp\(\)', 'CURRENT_TIMESTAMP', sql, flags=re.IGNORECASE)
sql = re.sub(r'curdate\(\)', 'CURRENT_DATE', sql, flags=re.IGNORECASE)
sql = re.sub(r'ON UPDATE CURRENT_TIMESTAMP', '', sql, flags=re.IGNORECASE)
sql = re.sub(r'enum\([^)]*\)', 'varchar(100)', sql, flags=re.IGNORECASE)
sql = re.sub(r"\s+COMMENT\s+'[^']*'", '', sql, flags=re.IGNORECASE)
print('Normalized identifiers, auto-increment, types, and comments.')

In [ ]:
sql = re.sub(r'longtext\s+CHARACTER SET\s+\S+\s+COLLATE\s+\S+\s+DEFAULT NULL\s+CHECK\s*\(json_valid\([^)]*\)\)', 'jsonb DEFAULT NULL', sql, flags=re.IGNORECASE)
sql = re.sub(r'longtext\s+CHARACTER SET\s+\S+\s+COLLATE\s+\S+', 'text', sql, flags=re.IGNORECASE)
sql = re.sub(r'\blongtext\b', 'text', sql, flags=re.IGNORECASE)
sql = re.sub(r'\)\s*ENGINE=[^;]+;', ');', sql, flags=re.IGNORECASE)
sql = re.sub(r'DEFAULT CHARSET=\S+', '', sql, flags=re.IGNORECASE)
sql = re.sub(r'COLLATE=\S+', '', sql, flags=re.IGNORECASE)
sql = re.sub(r'(?m)^\s*UNIQUE\s+KEY\s+\w+\s*\(([^)]*)\)\s*,?$', r'  UNIQUE (\1)', sql, flags=re.IGNORECASE)
sql = re.sub(r'(?m)^\s*KEY\s+\w+\s*\([^)]*\)\s*,?$', '', sql, flags=re.IGNORECASE)
sql = re.sub(r'(?m)^\s*DROP TABLE IF EXISTS (v_[A-Za-z0-9_]+);', r'DROP VIEW IF EXISTS \1;', sql)
sql = re.sub(r'(?m)^\s*DROP TABLE IF EXISTS (v_[A-Za-z0-9_]+)\s*;', r'DROP VIEW IF EXISTS \1;', sql)
print('Removed engine/charset, normalized JSON, keys, and view declarations.')

In [ ]:
final_sql = sql.strip() + '\n'
output_path.write_text(final_sql, encoding='utf-8')
schema_path.write_text(final_sql, encoding='utf-8')
print('Wrote converted SQL to:', output_path)
print('Original file overwritten with converted SQL.')

patterns = [
    r'`',
    r'ENGINE=',
    r'AUTO_INCREMENT',
    r'ON UPDATE CURRENT_TIMESTAMP',
    r'\benum\(',
    r'CHARACTER SET',
    r'COLLATE',
    r'json_valid',
    r'\bKEY\s+\w+\s*\(',
    r'COMMENT\s+\'',
    r'\blongtext\b',
    r'\btinyint\(1\)\b'
]
for pat in patterns:
    matches = re.findall(pat, final_sql, flags=re.IGNORECASE)
    print(f'{pat}:', len(matches))